# Day 6 - Basic Satellite Link Budget 

In this notebook, i will build a simple satellite communication link-budget calculator

The goal is to estimate received power, noise power, and link margin for a sateöllite-to-ground communication link.

## Link Budget Theory

A link budget estimates whether a communication link has enough received signal power compared with noise

The basic received power equation in dB form is: 

\[
P_r = P_t + G_t - L_{FS} - L_{other}
\]

where: 

- \(P_r\) is received power in dBW
- \(P_t\) is transmit power in dBW
- \(G_t\) is transmit antenna gain in dBi
- \(G_r\) is receive antenna gain in dBi
- \(L_{FS}\) is free-space path loss in dB
- \(L_{other}\) represents additional losses in dB

Noise power is:

\[
N = kTB
\]

where:

- \(k\) is Boltzmann's constant
- \(T\) is system noise temperature in kelvin
- \(B\) is receiver bandwidth in Hz


The signal-to-noise ratio is:

\[
SNR_{dB} = P_{r,dBW} - N_{dBW}
\]

The link margin is:

\[
Margin = SNR_{dB} - SNR_{required,dB}
\]

if margin is positive, the simplified link is considered usable.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path # help us saving plot files cleanly

In [4]:
SPEED_OF_LIGHT_M_PER_S = 299_792_458
BOLTZMANN_CONSTANT_J_PER_K = 1.380649e-23
# we need the bolzmann constant to calculate thermal noise power

In [ ]:
# Helper Functions
def watts_to_dbw(power_watts):
    """
    Conver power from watts to dBW

    dBW means decibels relative to 1 watt

    Example: 
    1W = 0 dBW
    10W = 10 dBW
    """
    power_dbw = 10 * np.log(power_watts)
    return power_dbw

# this function converts watts into dBW, this function is why dB units are convenient: multiplication and division become addition and subtraction

def dbw_to_watts(power_dbw):
    """
    Convert power from dBW to watts
    """
    power_watts = 10 ** (power_dbw / 10)
    return power_watts

# this is inverse conversion, the equation: Pw = 10^(P_dBW/10)

def calculate_fspl_db(range_m, frequency_hz):
    """
    Calculate free-space path loss in dB
    """
    fspl_db = 20 * np.log10(4 * np.pi * range_m * frequency_hz) / SPEED_OF_LIGHT_M_PER_S
    return fspl_db

# free space path loss formula is used for lin budget calculation again which needs path loss.

def calculate_received_power_dbw(transmit_power_dbw, transmit_antenna_gain_dbi, receive_antenna_gain_dbi, fspl_db, other_losses_db,):
    """
    Calculate received power in dBW using a simple link-budget equation 
    """
    received_power_dbw = transmit_power_dbw + transmit_antenna_gain_dbi + receive_antenna_gain_dbi - fspl_db - other_losses_db

    return received_power_dbw

# received power = transmit power + gains - losses
# a negative dBW value is normal. Satellite received signals are usually very weak

def calculate_noise_power_dbw(system_noise_temperature_k, bandwidth_hz):
    """
    Calculate thermal noise power in dBW
    """
    noise_power_watts = (BOLTZMANN_CONSTANT_J_PER_K * system_noise_temperature_k * bandwidth_hz)

    noise_power_dbw = watts_to_dbw(noise_power_watts)

    return noise_power_dbw

    # this calculates thermal noise power 
    # Noise increases when: 
    # system noise temperature increases 
    # bandwith increases
    # wider receiver bandwidth collects more noise 
    # important in satcom

def calculate_snr_db(received_power_dbw, noise_power_dbw):

    """
    Calculate signal-to-noise ratio in dB
    """

    snr_db = received_power_dbw - noise_power_dbw
    return snr_db

# example received_power = -101 dBW
# noise power = -142 dBW
# SNR = -101 - (-142) = 41 dB
# because both values are in dBW, substracting them gives SNR in dB

def calculate_link_margin_db(snr_db, required_snr_db):

    """
    Calculate link margin in dB
    """
    link_margin_db = snr_db - required_snr_db
    return link_margin_db

#
